In [ ]:
import numpy as np 
import numpy.ma as ma
import pandas as pd 
import os
import finlab
from finlab import data
import matplotlib.pyplot as plt
import json

finlab_token = os.getenv("FINLAB_API_KEY")
finlab.login(finlab_token)


輸入成功!


In [3]:
VALID_DAYS = 270
MAX_STOCK_PER_DAY = 1
ROI_STOCK_PER_DAY = 1
PRED_LEN = 30
TAX_RATE = 0.003
FEE = 0.001425
THRESHOLD = 0.5
GROUPGAP = 30

In [4]:
from metrics.correlations import daily_correlation, stockly_correlation
from metrics.precisions import top_ratio_overlap_rate, top_ratio_overlap_rate_daily, precision_at_ratio, precision_daily_at_ratio, precision_baseline, get_top_n_avg
from metrics.intervals import calculate_precision, extract_interval_matches, calculate_total_truth, group_intervals, get_prediction_interval_roi
from metrics.loss import mse_loss

In [5]:
df_close = data.get("etl:adj_close")

Your version is 1.2.20, please install a newer version.
Use "pip install finlab==1.3.0" to update the latest version.


In [6]:

def positive(x):
    return x > 0.3

def analyze(pred_df, truth_df):
    pred_values = pred_df.values.flatten()
    truth_values = truth_df.values.flatten()
    
    corr = ma.corrcoef(ma.masked_invalid(pred_values), ma.masked_invalid(truth_values))[0][1]
    daily_corr = daily_correlation(pred_df, truth_df)
    stockly_corr = stockly_correlation(pred_df, truth_df)
    mse = mse_loss(pred_values, truth_values)
    
    pred_df = pred_df.fillna(-1)
    
    threshold = 0.5
    result = extract_interval_matches(pred_df, truth_df, threshold, group_gap=GROUPGAP)
    TP, total_positive, total_truth = calculate_precision(result)
    truth_intervals_dict, total_truth = calculate_total_truth(truth_df, threshold, GROUPGAP)
    
        
    precision_1 = precision_at_ratio(pred_df, truth_df, 0.01, positive)
    precision_daily_1 = precision_daily_at_ratio(pred_df, truth_df, 0.01, positive)
    precision_5 = precision_at_ratio(pred_df, truth_df, 0.05, positive)
    precision_daily_5 = precision_daily_at_ratio(pred_df, truth_df, 0.05, positive)
    precision_10 = precision_at_ratio(pred_df, truth_df, 0.1, positive)
    precision_daily_10 = precision_daily_at_ratio(pred_df, truth_df, 0.1, positive)
    baseline_precision = precision_baseline(truth_df, positive)
    
    return {
        "corr": corr,
        "daily_corr": daily_corr,
        "stockly_corr": stockly_corr,
        "TP": TP,
        "total_positive": total_positive,
        "total_truth": total_truth,
        "mse": mse,
        "precision@0.01": precision_1,
        "precision@0.05": precision_5,
        "precision@0.1": precision_10,
        "precision_daily@0.01": precision_daily_1,
        "precision_daily@0.05": precision_daily_5,
        "precision_daily@0.1": precision_daily_10,
        "baseline_precision": baseline_precision,
    }
        
def get_one_directory_analysis(result_dir):
    args_path = os.path.join(result_dir, "args.json")
    with open(args_path, 'r') as f:
        args = json.load(f)
    
    
    print(f"Processing directory: {result_dir}")
    test_pred_df, test_truth_df, val_pred_df, val_truth_df = read_all_df(result_dir)
    analysis = analyze(test_pred_df, test_truth_df)
    
    used_keys = ['category', 'broker', 'loss', 'goal']
    args = {k: args[k] for k in used_keys if k in args}
    
    data_row = args.copy()
    data_row.update(analysis)
    
    return data_row

def get_directory_analysis(result_root_dir):
    print(f"Analyzing results in directory: {result_root_dir}")
    filenames = os.listdir(result_root_dir)
    print(f"Found {len(filenames)} files/directories in the root directory.")
    result_datas = []

    for filename in filenames:
        # if 'max_price' in filename and 'ipynb' not in filename:
        if not os.path.isdir(os.path.join(result_root_dir, filename)):
            continue

        result_dir = os.path.join(result_root_dir, filename)
        data_row = get_one_directory_analysis(result_dir)
        result_datas.append(data_row)

    result_df = pd.DataFrame(result_datas)
    return result_df

In [8]:
get_one_directory_analysis('../results/Top50')

FileNotFoundError: [Errno 2] No such file or directory: '../results/Top50/args.json'

In [7]:
from unittest import result


used_categories = ['Top50', 'Top100',  'ChemicalTSE', 'FinanceTSE', 'OptoTSE', "SemiconductorTSE",
                  "Selected", "SelectedVer2", "CarTSE", "NetworkTSE", "EETSE"]

ROOT_DIR = '../../results_all_categories_2'

result_datas = []

for category in used_categories:
    for broker in [0, 1]:
        result_dir = f"{ROOT_DIR}/{category}_{broker}_ccc"
        data_row = get_one_directory_analysis(result_dir)
        result_datas.append(data_row)

result_df = pd.DataFrame(result_datas)

Processing directory: ../../results_all_categories_2/Top50_0_ccc
Processing directory: ../../results_all_categories_2/Top50_1_ccc
Processing directory: ../../results_all_categories_2/Top100_0_ccc
Processing directory: ../../results_all_categories_2/Top100_1_ccc
Processing directory: ../../results_all_categories_2/ChemicalTSE_0_ccc
Processing directory: ../../results_all_categories_2/ChemicalTSE_1_ccc
Processing directory: ../../results_all_categories_2/FinanceTSE_0_ccc
Processing directory: ../../results_all_categories_2/FinanceTSE_1_ccc
Processing directory: ../../results_all_categories_2/OptoTSE_0_ccc
Processing directory: ../../results_all_categories_2/OptoTSE_1_ccc
Processing directory: ../../results_all_categories_2/SemiconductorTSE_0_ccc
Processing directory: ../../results_all_categories_2/SemiconductorTSE_1_ccc
Processing directory: ../../results_all_categories_2/Selected_0_ccc
Processing directory: ../../results_all_categories_2/Selected_1_ccc
Processing directory: ../../result

In [7]:
used_categories = ['Top50', 'Top100',  'ChemicalTSE', 'FinanceTSE', 'OptoTSE', "SemiconductorTSE",
                  "Selected", "SelectedVer2", "CarTSE", "NetworkTSE", "EETSE"]

ROOT_DIR = '../../results_all_categories_2'

result_datas = []

for category in used_categories:
    result_dir = f"{ROOT_DIR}/{category}_0_ccc"
    df, _, _, _ = read_all_df(result_dir)
    print(category, len(df.columns))

Top50 49
Top100 97
ChemicalTSE 26
FinanceTSE 33
OptoTSE 58
SemiconductorTSE 70
Selected 131
SelectedVer2 129
CarTSE 30
NetworkTSE 37
EETSE 42


In [12]:
result_df = result_df.sort_values(by='category', ascending=False).round(4)
result_df.to_csv("tmp/result_analysis.csv", index=False)
result_df.groupby('broker').aggregate(
    {
        'corr': 'mean',
        'daily_corr': 'mean',
        'stockly_corr': 'mean',
        'TP': 'sum',
        'total_positive': 'sum',
        'total_truth': 'sum',
        'mse': 'mean',
        'precision@0.01': 'mean',
        'precision@0.05': 'mean',
        'precision@0.1': 'mean',
        'precision_daily@0.01': 'mean',
        'precision_daily@0.05': 'mean',
        'precision_daily@0.1': 'mean',
        'baseline_precision': 'mean'
    }
)

,corr,daily_corr,stockly_corr,TP,total_positive,total_truth,mse,precision@0.01,precision@0.05,precision@0.1,precision_daily@0.01,precision_daily@0.05,precision_daily@0.1,baseline_precision
broker,,,,,,,,,,,,,,
0,0.201845,0.202064,0.102891,94,480,652,0.021245,0.231191,0.164273,0.134664,0.179555,0.165900,0.132391,0.050182
1,0.197564,0.196282,0.099282,98,493,652,0.021845,0.219618,0.161182,0.130273,0.175900,0.161255,0.129800,0.050182


In [10]:
test_pred_df, test_truth_df, val_pred_df, val_truth_df = read_all_df("../../results_all_categories_2/SelectedVer2_0_ccc")

# 假設 test_pred_df、test_truth_df 已經是兩個相同日期索引、相同欄位的 DataFrame
for n in range(1, 5):
    avg_true = get_top_n_avg(test_pred_df, test_truth_df, n)
    print(f"Average true value of top-predicted stock each day: {avg_true:.4f}")

baseline = test_truth_df.mean(axis=0).mean(axis=0)
print(f"{baseline}")
print(f"")


Average true value of top-predicted stock each day: 0.1574
Average true value of top-predicted stock each day: 0.1530
Average true value of top-predicted stock each day: 0.1508
Average true value of top-predicted stock each day: 0.1484
0.0848451414456924



In [ ]:
result_df = get_directory_analysis("../../results_all_categories_2")
result_df = result_df.sort_values(by='category')
result_df

,category,broker,loss,goal,corr,daily_corr,stockly_corr,TP,total_positive,total_truth,mse,precision@0.01,precision@0.05,precision@0.1,precision_daily@0.01,precision_daily@0.05,precision_daily@0.1,baseline_precision
9,BioMedTSE,0,mse,max_price,0.184228,0.188089,0.089492,2,12,31,0.022800,0.234694,0.164291,0.137767,0.160695,0.160695,0.131741,0.052083
20,BioMedTSE,0,ccc,max_price,0.123382,0.126604,0.035430,5,23,31,0.026971,0.221088,0.159538,0.131659,0.162866,0.162866,0.134274,0.052083
11,BuildingTSE,0,mse,max_price,0.166167,0.173232,0.081814,3,18,37,0.016404,0.213280,0.112228,0.094309,0.134636,0.116178,0.079479,0.041642
6,BuildingTSE,0,ccc,max_price,0.146641,0.171379,0.054119,2,17,37,0.017545,0.154930,0.123894,0.097929,0.138979,0.114007,0.092508,0.041642
23,CarTSE,0,mse,max_price,0.161045,0.181104,0.103917,2,9,27,0.017637,0.239130,0.130340,0.091929,0.124864,0.124864,0.098082,0.052515
3,CarTSE,0,ccc,max_price,0.168192,0.172029,0.095136,2,13,27,0.019744,0.231884,0.127444,0.095548,0.123779,0.123779,0.103873,0.052515
2,ChemicalTSE,0,ccc,max_price,0.218015,0.251777,0.099788,7,13,22,0.023436,0.334728,0.203008,0.166249,0.236699,0.236699,0.163409,0.052410
30,ChemicalTSE,0,mse,max_price,0.156628,0.208820,0.045196,3,13,22,0.023486,0.217573,0.152882,0.116124,0.158523,0.158523,0.133550,0.052410
19,ClothesTSE,0,mse,max_price,0.207880,0.224816,0.153533,1,9,21,0.009354,0.114804,0.071213,0.058522,0.067318,0.067318,0.052479,0.023797
29,ClothesTSE,0,ccc,max_price,0.145282,0.154470,0.100559,1,12,21,0.011721,0.072508,0.044659,0.034389,0.024973,0.024973,0.026421,0.023797
